In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col


import constants as c
import helpers as h 
from logger import setup_logger 
log = setup_logger('added-population-coverage')
log.setLevel('INFO')
log.info("Modules loaded.")

2026-02-06 13:17:26 - added-population-coverage - INFO - Modules loaded.


In [2]:
analysis_df = pd.read_csv(c.CURRENT_DF)
analysis_df = h.add_covariate_cols(analysis_df)
analysis_df = h.add_demo_cols(analysis_df)
analysis_df = h.add_estimate_cols(analysis_df)


2026-02-06 13:17:26 - analysis-helpers - INFO - Found 192 tracts with at least one FloodNet sensor.
2026-02-06 13:17:26 - analysis-helpers - INFO - Found 2171 311 requests.
2026-02-06 13:17:26 - analysis-helpers - INFO - Found 878 tracts with at least one 311 report.
2026-02-06 13:17:26 - analysis-helpers - INFO - Found 1001 tracts with no DEP flooding.
2026-02-06 13:17:26 - analysis-helpers - INFO - Added fraction white (frac_white) column.
2026-02-06 13:17:26 - analysis-helpers - INFO - Added fraction black (frac_black) column.
2026-02-06 13:17:26 - analysis-helpers - INFO - Added fraction hispanic (frac_hispanic) column.
2026-02-06 13:17:26 - analysis-helpers - INFO - Added fraction asian (frac_asian) column.
2026-02-06 13:17:26 - analysis-helpers - INFO - Added fraction high school graduates (frac_hs) column.
2026-02-06 13:17:26 - analysis-helpers - INFO - Added fraction bachelors degree (frac_bachelors) column.
2026-02-06 13:17:26 - analysis-helpers - INFO - Added fraction graduat

In [3]:
EST_TO_USE = c.ESTIMATE_TO_USE
log.info(f"Using estimate: {EST_TO_USE}")

2026-02-06 13:17:26 - added-population-coverage - INFO - Using estimate: confirmed_or_above_thres


# basic exploratory analysis

In [4]:
pd.set_option('display.max_columns', 500)
analysis_df.head()


,BoroName,BoroCT2020,NTAName,CDTANAME,PUMA,empirical_estimate,p_y,p_y_CI_lower,p_y_CI_upper,n_images_by_area,empirical_estimate_p_alop,at_least_one_positive_image_by_area,at_least_one_positive_image_by_area_CI_lower,at_least_one_positive_image_by_area_CI_upper,total_population,nhl_white_alone,nhl_black_alone,hispanic_alone,nhl_asian_alone,n_children,n_elderly,total_households,num_households_with_internet,num_households_with_smartphone,median_household_income,num_high_school_graduates,num_bachelors_degree,num_graduate_degree,num_limited_english_speaking_households,ft_elevation_min,ft_elevation_max,ft_elevation_mean,area,n_floodnet_sensors,n_catch_basins,catch_basin_density,cb_days_clogged,has_clogged_cb_complaint,cb_avg_resolution_time,dep_moderate_1_area,dep_moderate_1_frac,dep_moderate_2_area,dep_moderate_2_frac,GEOID,sewer_backup_311c,street_flooding_311c,catch_basin_clogged/flooding_311c,manhole_overflow_311c,highway_flooding_311c,any_sensors,n_311_reports,any_311_report,no_dep_flooding,frac_white,frac_black,frac_hispanic,frac_asian,frac_hs,frac_bachelors,frac_grad,frac_children,frac_elderly,frac_internet,frac_smartphone,frac_limited_english,confirmed_flooding,above_thres,confirmed_or_above_thres
0,Manhattan,1000100,The Battery-Governors Island-Ellis Island-Libe...,MN01 Financial District-Tribeca (CD 1 Equivalent),4121,NaN,0.050211,2.158959e-314,1.000000,0,NaN,0.000000,0.000000,0.000000,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0.0,19.0,8.153329,1.842909e+06,0.0,0.0,0.000000,0.0,0,0.000000,0.000000,0.000000,0.000000,0.000000,36061000100,0,0,0,0,0,False,0,False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True,True
1,Manhattan,1001401,Lower East Side,MN03 Lower East Side-Chinatown (CD 3 Equivalent),4103,0.0,0.001444,9.342801e-06,0.009392,259,0.0,0.210764,0.002417,0.913187,3395,2371,65,370,258,328,1311,1761,1471,1385,94076.0,398,930,790,100,24.0,43.0,34.847163,1.006117e+06,0.0,13.0,0.000013,0.0,0,0.000000,0.000000,0.000000,0.000000,0.000000,36061001401,1,0,0,0,0,False,1,True,True,0.698380,0.019146,0.108984,0.075994,0.117231,0.273932,0.232695,0.096613,0.386156,0.835321,0.786485,0.056786,False,False,False
2,Manhattan,1001402,Lower East Side,MN03 Lower East Side-Chinatown (CD 3 Equivalent),4103,0.0,0.000461,1.630542e-06,0.003213,705,0.0,0.183762,0.001149,0.896538,3449,1008,249,813,1166,249,910,1807,1520,1584,50347.0,311,1014,360,647,13.0,42.0,27.553037,1.226207e+06,0.0,44.0,0.000036,0.0,0,0.000000,3811.632650,0.003108,7439.195282,0.006067,36061001402,0,0,0,0,0,False,0,False,False,0.292259,0.072195,0.235720,0.338069,0.090171,0.293998,0.104378,0.072195,0.263845,0.841173,0.876591,0.358052,False,False,False
3,Manhattan,1001800,Lower East Side,MN03 Lower East Side-Chinatown (CD 3 Equivalent),4103,0.0,0.000220,2.648354e-06,0.001355,1783,0.0,0.228073,0.004711,0.910932,6702,2280,410,905,2921,336,1408,2979,2647,2731,79973.0,536,2252,557,428,8.0,45.0,36.394639,2.399278e+06,0.0,94.0,0.000039,8.0,1,2.666667,0.000000,0.000000,0.000000,0.000000,36061001800,0,0,0,0,0,False,0,False,True,0.340197,0.061176,0.135034,0.435840,0.079976,0.336019,0.083110,0.050134,0.210087,0.888553,0.916751,0.143672,False,False,False
4,Manhattan,1002201,Lower East Side,MN03 Lower East Side-Chinatown (CD 3 Equivalent),4103,0.0,0.001083,1.971842e-05,0.006184,422,0.0,0.262862,0.008287,0.927023,6437,1755,1453,1983,977,988,1281,3011,2232,2662,38594.0,1283,968,580,368,9.0,28.0,17.288704,1.740175e+06,0.0,34.0,0.000020,0.0,1,0.000000,14976.604512,0.008606,16900.033063,0.009712,36061002201,1,0,0,0,0,False,1,True,False,0.272643,0.225726,0.308063,0.151779,0.199316,0.150381,0.090104,0.153488,0.199006,0.741282,0.884092,0.122219,False,False,False


In [5]:
print(analysis_df['total_population'].isna().sum())

0


In [6]:
analysis_df[['dep_moderate_1_area', 'dep_moderate_2_area']].describe()

,dep_moderate_1_area,dep_moderate_2_area
count,2325.000000,2.325000e+03
mean,34125.087468,5.184025e+04
std,80743.105572,1.265335e+05
min,0.000000,0.000000e+00
25%,0.000000,0.000000e+00
50%,5253.617806,7.093228e+03
75%,33654.810667,4.673852e+04
max,994791.061756,1.731771e+06


In [7]:
analysis_df['no_dep_flooding'] = (analysis_df['dep_moderate_1_area'] == 0) & (analysis_df['dep_moderate_2_area'] == 0)
print("Population in these locations: %2.3f" % analysis_df.loc[(analysis_df[EST_TO_USE] == 1) & (analysis_df['no_dep_flooding'] == 1), 'total_population'].sum())

Population in these locations: 268197.000


# 311

### still, our model identifies lots of high-risk areas with no 311 reports!

In [8]:
print("Population in these locations: %2.3f" % analysis_df.loc[(analysis_df[EST_TO_USE] == 1) & (analysis_df['any_311_report'] == 0), 'total_population'].sum())

Population in these locations: 401221.000


# flood sensors

In [9]:
print("Population in these locations: %2.3f" % analysis_df.loc[(analysis_df[EST_TO_USE]) & (analysis_df['any_sensors'] == 0), 'total_population'].sum())

Population in these locations: 899434.000


In [10]:
analysis_df['n_floodnet_sensors'].sum()

np.float64(253.0)

### Other stats 

In [11]:
# population in tracts with no other coverage except from model risk 
print("Population in these locations: %2.3f" % analysis_df.loc[(analysis_df[EST_TO_USE]) & (analysis_df['any_311_report'] == 0) & (analysis_df['any_sensors'] == 0) & (analysis_df['no_dep_flooding']), 'total_population'].sum())

Population in these locations: 119680.000


In [12]:
# total population in tracts classified as high risk 
print("Population in these locations: %2.3f" % analysis_df.loc[(analysis_df[EST_TO_USE]), 'total_population'].sum())

Population in these locations: 1109445.000


In [13]:
EST_COLS = ['above_thres', 'confirmed_flooding', 'confirmed_or_above_thres']
# generate a table of population coverage (same as above) for each estimate

def get_pop_coverage(df, est_col):
    pop_in_risk = df.loc[(df[est_col]), 'total_population'].sum()
    pop_in_risk_no_dep = df.loc[(df[est_col]) & (df['no_dep_flooding'] == 1), 'total_population'].sum()
    pop_in_risk_no_311 = df.loc[(df[est_col]) & (df['any_311_report'] == 0), 'total_population'].sum()
    pop_in_risk_no_sensors = df.loc[(df[est_col]) & (df['any_sensors'] == 0), 'total_population'].sum()
    pop_in_risk_no_other = df.loc[(df[est_col]) & (df['any_311_report'] == 0) & (df['any_sensors'] == 0), 'total_population'].sum()
    pop_in_risk_no_other_no_dep = df.loc[(df[est_col]) & (df['any_311_report'] == 0) & (df['any_sensors'] == 0) & (df['no_dep_flooding']), 'total_population'].sum()
    return pop_in_risk, pop_in_risk_no_dep, pop_in_risk_no_311, pop_in_risk_no_sensors, pop_in_risk_no_other, pop_in_risk_no_other_no_dep

# informative col names 
cols = {
    'above_thres': 'Above Threshold',
    'confirmed_flooding': 'Confirmed Flooding',
    'confirmed_or_above_thres': 'Confirmed or Above Threshold'
}

rows = {
    'pop_in_risk': 'Population in Flooded Tracts',
    'pop_in_risk_no_dep': 'Population in Flooded Tracts with No DEP Flooding',
    'pop_in_risk_no_311': 'Population in Flooded Tracts with No 311 Reports',
    'pop_in_risk_no_sensors': 'Population in Flooded Tracts with No FloodNet Sensors',
    'pop_in_risk_no_other': 'Population in Flooded Tracts with No Other Coverage',
    'pop_in_risk_no_other_no_dep': 'Population in Flooded Tracts with No Other Coverage and No DEP Flooding'
}

# make a table 
pop_coverage = pd.DataFrame(index=rows.values(), columns=cols.values())
for est_col in EST_COLS:
    pop_coverage.loc[:, cols[est_col]] = get_pop_coverage(analysis_df, est_col)

# format nicely with commas 
pop_coverage = pop_coverage.applymap(lambda x: "{:,.0f}".format(x))
pop_coverage


/tmp/ipykernel_907287/3048955306.py:35: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pop_coverage = pop_coverage.applymap(lambda x: "{:,.0f}".format(x))


,Above Threshold,Confirmed Flooding,Confirmed or Above Threshold
Population in Flooded Tracts,"931,970","600,698","1,109,445"
Population in Flooded Tracts with No DEP Flooding,"225,083","103,593","268,197"
Population in Flooded Tracts with No 311 Reports,"286,709","232,019","401,221"
Population in Flooded Tracts with No FloodNet Sensors,"762,318","461,760","899,434"
Population in Flooded Tracts with No Other Coverage,"271,370","214,681","368,550"
Population in Flooded Tracts with No Other Coverage and No DEP Flooding,"97,827","45,003","119,680"


# sensitivity analysis: different thresholds for high flood risk

In [14]:
# Recompute results for 10th and 50th percentile thresholds
thresholds = [0.10, 0.50]
results = {}

confirmed_p_y = analysis_df[analysis_df['confirmed_flooding']]['p_y']

for q in thresholds:
    q_val = confirmed_p_y.quantile(q)
    col_name = f'above_thres_{int(q*100)}'
    analysis_df[col_name] = analysis_df['p_y'] > q_val
    
    # Also create the combined 'confirmed or above threshold' column for this quantile
    combined_col = f'confirmed_or_above_thres_{int(q*100)}'
    analysis_df[combined_col] = analysis_df['confirmed_flooding'] | analysis_df[col_name]
    
    # Compute coverage for this combined estimate
    results[f'{int(q*100)}th Percentile'] = get_pop_coverage(analysis_df, combined_col)

# Add the original (25th percentile) results for comparison
results['25th Percentile (Original)'] = get_pop_coverage(analysis_df, 'confirmed_or_above_thres')

# Create comparison table
comparison_df = pd.DataFrame(results, index=rows.values())
comparison_df = comparison_df.applymap(lambda x: "{:,.0f}".format(x))
comparison_df

/tmp/ipykernel_907287/1090517023.py:24: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  comparison_df = comparison_df.applymap(lambda x: "{:,.0f}".format(x))


,10th Percentile,50th Percentile,25th Percentile (Original)
Population in Flooded Tracts,"1,607,567","778,480","1,109,445"
Population in Flooded Tracts with No DEP Flooding,"431,092","164,049","268,197"
Population in Flooded Tracts with No 311 Reports,"603,943","291,767","401,221"
Population in Flooded Tracts with No FloodNet Sensors,"1,328,961","614,844","899,434"
Population in Flooded Tracts with No Other Coverage,"538,414","274,429","368,550"
Population in Flooded Tracts with No Other Coverage and No DEP Flooding,"174,261","63,976","119,680"
